# 📊 Hệ Thống RAG Hỏi Đáp Báo Cáo Tài Chính (Financial Multimodal RAG)

**Môi trường thực thi:** Google Colab - GPU T4 (15GB/16GB VRAM)  
**Model LLM:** `Qwen/Qwen2.5-3B-Instruct` (fp16)  
**Embedding & Index:** `BAAI/bge-m3` + FAISS + SQLite FTS5 (BM25)  
**Cơ sở dữ liệu số liệu có cấu trúc:** SQLite (`company`, `report`, `metric`, `fact`, `chunk`, `chunk_fts`)  
**Giao diện Demo:** Gradio Web UI (`share=True` tạo link public để truy cập trực tiếp từ trình duyệt)  

---
### ⚠️ Lưu ý quan trọng trước khi chạy:
- Vào menu **Runtime → Change runtime type → Chọn T4 GPU**.
- Notebook đã được cấu hình tự động clone mã nguồn, cài đặt dependencies tương thích CUDA 12, seed dữ liệu và chạy Qwen2.5-3B fp16 hoàn toàn trên GPU T4.

## 1. Kiểm tra phần cứng GPU (Hardware Check)

In [ ]:
# Kiểm tra GPU T4 và CUDA trên Colab
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Bạn chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU.")


## 2. Chuẩn bị Mã nguồn & Cài đặt Dependencies

Tự động clone repository từ GitHub (nếu chưa có) và cài đặt các thư viện tương thích với Colab CUDA 12.

In [ ]:
import os
import sys

# 1. Clone repository vào /content/financial-rag nếu chưa tồn tại
REPO_DIR = "/content/financial-rag"
if not os.path.exists(REPO_DIR):
    !git clone -b Thien https://github.com/dongdx2k5/Financial-Multimodal-RAG.git {REPO_DIR}

# 2. Chuyển thư mục làm việc vào repo
%cd {REPO_DIR}

# 3. Thêm src vào sys.path để import ổn định trong Colab kernel
src_path = os.path.join(REPO_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# 4. Cài đặt gói financial-rag và các thư viện cần thiết (dùng faiss-cpu để tránh lỗi CUDA 12 của faiss-gpu)
!pip install -q -e .
!pip install -q transformers accelerate sentence-transformers faiss-cpu gradio


## 3. Khởi tạo Cơ sở dữ liệu & Nạp Báo cáo Tài chính (Seeding & Ingestion)

- Khởi tạo SQLite database, các bảng chỉ tiêu và bảng FTS5.
- Nạp các báo cáo cốt lõi (AAA, NVDA, HPG, VNM) trích xuất dữ liệu có cấu trúc và tạo index vector BGE-M3.

In [ ]:
# Khởi tạo schema và từ điển chỉ tiêu Việt - Anh
!python -m financial_rag.cli seed

# Nạp và tạo index dữ liệu
!python -m financial_rag.cli ingest

# Kiểm tra chẩn đoán hệ thống (Doctor)
!python -m financial_rag.cli doctor


## 4. Chạy Đánh Giá Benchmark (Evaluation)

Đánh giá trên 36 câu hỏi chuẩn gồm 5 nhóm:
1. Tra cứu số liệu đơn lẻ (Lookup)
2. Tính toán chỉ số tài chính (Calculation: biên lợi nhuận, D/E, tăng trưởng)
3. Giải thích chính sách và thuyết minh (Text explanation)
4. So sánh giữa các doanh nghiệp (Comparison)
5. Câu hỏi bẫy ngoài phạm vi / dự báo cổ phiếu (Trap refusal)

In [ ]:
# Chạy đánh giá nhanh (chế độ Mock giúp kiểm thử toàn bộ pipeline logic trong ~25 giây)
!LLM_BACKEND=mock python -m financial_rag.cli evaluate


## 5. Khởi chạy Giao diện Gradio Web UI với Qwen2.5-3B-Instruct

- Tự động nạp mô hình `Qwen/Qwen2.5-3B-Instruct` ở định dạng fp16 lên GPU T4 (~6.2GB VRAM).
- Kết hợp embedding `BAAI/bge-m3` (~2.2GB VRAM) trên GPU T4 để đạt tốc độ phản hồi tối ưu nhất.
- Gradio chạy với `share=True` để cung cấp đường link public (`https://xxxx.gradio.live`) dễ dàng demo.

In [ ]:
import os
import gc
import torch

# Giải phóng VRAM trước khi load model để tránh tràn bộ nhớ nếu chạy lại cell
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

# Cấu hình biến môi trường chạy GPU T4
os.environ["LLM_BACKEND"] = "qwen_local"
os.environ["LLM_MODEL_ID"] = "Qwen/Qwen2.5-3B-Instruct"
os.environ["LLM_DEVICE"] = "cuda"
os.environ["LLM_FP16"] = "true"
os.environ["EMBEDDING_MODEL"] = "BAAI/bge-m3"
os.environ["EMBEDDING_DEVICE"] = "cuda"

from financial_rag.config import settings
from financial_rag.db.repository import FinancialRepository
from financial_rag.reasoning.engine import FinancialRAGEngine
from financial_rag.llm.model import LLMBackend
from financial_rag.ui.app import create_gradio_app

# Khởi tạo Repository & Engine
db_p = settings.resolve_path(settings.database_path)
repo = FinancialRepository(db_p)
llm = LLMBackend(backend_type="qwen_local", model_id="Qwen/Qwen2.5-3B-Instruct", device="cuda")
engine = FinancialRAGEngine(repo=repo, llm=llm)

print("\n✅ Hệ thống đã sẵn sàng! Đang khởi tạo Gradio Web UI...")
print("👉 Nhấp vào đường link 'Running on public URL: https://....gradio.live' bên dưới để trải nghiệm.\n")

# Launch Gradio với link public
app = create_gradio_app(engine=engine, repo=repo)
app.launch(share=True, debug=False)


## 6. Hỏi đáp trực tiếp bằng dòng lệnh (CLI Quick Test)

Bạn cũng có thể đặt câu hỏi trực tiếp mà không cần mở giao diện Gradio.

In [ ]:
# Ví dụ tra cứu nhanh bằng lệnh finrag ask
!python -m financial_rag.cli ask "Doanh thu thuần năm 2025 của Công ty Nhựa An Phát Xanh (AAA) là bao nhiêu?"
